In [1]:
from f2fmatcher.io import czi_reader
from f2fmatcher.segmentation import cellpose_seg
import os, sys, multiprocessing
from tqdm import tqdm
import pandas as pd
import numpy as np
from PIL import Image   
from scipy.ndimage import binary_dilation, binary_erosion
from skimage.morphology import disk
from matplotlib import pyplot as plt

sys.path.append("/DATA/F2FMatcher_DDC")
from config.ddc_config import *


In [2]:
img = "22-082_10X_DAPI_LAM_DYS_COL4_7-Scene-1-QUAG26"
muscle = "QUA"
slide = 1
channel = 2
dir_save_png = Path("./")
n_process = 32

if muscle == "TA":
    dir_czi_source = CZI_BASE_DIR_TA
    dir_CP_MASKS = CP_MASKS_DIR_TA
    dir_pair_output = PAIR_DIRS_BASE_TA
elif muscle == "QUA":
    dir_czi_source = CZI_BASE_DIR_QUA
    dir_CP_MASKS = CP_MASKS_DIR_QUA
    dir_pair_output = PAIR_DIRS_BASE_QUA

IHF = SLIDES[slide]["IHF"]
staining = SLIDES[slide]["stainings"][channel]
if IHF:
    param_img = ("fluorescence", SLIDES[slide]["scanning_objective"], 1.0)
else:
    param_img = ("brightfield", SLIDES[slide]["scanning_objective"], 1.0)

dir_czi = dir_czi_source / f'{SLIDES[slide]["czi_dir"]}' / f'{img}.czi'
dir_png = dir_save_png / f'{img}.png'

CP_model_name = SLIDES[slide]["segmentation_model"]
CP_model_path = Path("/DATA/F2FMatcher/models/CellPose2_finetuned")

In [3]:
# import czi, resize and export as png
czi_reader.import_resize_export_czi(
    czi_path=dir_czi,
    IHF=IHF,
    channel_index=channel,
    dir_save_png=dir_save_png,
    param_img=param_img,
)

OSError: [Errno 112] Host is down: '/media/DATABRUT/DB_DDC/serverGPU/AJ/22-082_QUA/IHF_Lam-Dys-Col4/22-082_10X_DAPI_LAM_DYS_COL4_7-Scene-1-QUAG26.czi'

In [ ]:
masks, props, _ = cellpose_seg.get_CP_masks(
    img_path=dir_png, 
    CP_model_name=CP_model_name, 
    CP_model_path=CP_model_path,
    savedir=dir_CP_MASKS, 
    channels = [0, 0],
)

# filter small fibers
region_labels = {int(r.label):[r.bbox, r.area] for r in props if r.area >= threshold_fiber_area}

OutOfMemoryError: CUDA out of memory. Tried to allocate 196.00 MiB. GPU 0 has a total capacity of 23.60 GiB of which 103.88 MiB is free. Process 826305 has 20.84 GiB memory in use. Including non-PyTorch memory, this process has 2.64 GiB memory in use. Of the allocated memory 2.12 GiB is allocated by PyTorch, and 264.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# open resized PNG image
image = np.array(Image.open(dir_png))
assert masks.shape == image.shape
y_full, x_full = masks.shape

if not IHF:
    # convert to grayscale with black background if brightfield
    image = 255 - image

In [ ]:
def dilate_mask(mask_i, n_px):
    return binary_dilation(mask_i, structure=disk(n_px)).astype(np.uint8)

def erode_mask(mask_i, n_px):
    return binary_erosion(mask_i, structure=disk(n_px)).astype(np.uint8)

def crop_image_with_bbox(image, bbox_i):
    y_min, x_min, y_max, x_max = bbox_i
    image_i = image[y_min:y_max, x_min:x_max]
    return image_i

def compute_features_channe(mask_i, image_i):
    """
    Calculate intensity statistics for each compartment.
    """
    if mask_i.sum() == 0:
        stats = {
            'mean': np.nan,
            'std': np.nan,
            'p10': np.nan,
            'p25': np.nan,
            'p50': np.nan,
            'p75': np.nan,
            'p90': np.nan,
            'skew': np.nan,
            'kurt': np.nan
        }
    else:
        pixels = image_i[mask_i]
        stats = {
            'mean': np.mean(pixels),
            'std': np.std(pixels),
            'p10': np.percentile(pixels, 10),
            'p25': np.percentile(pixels, 25),
            'p50': np.percentile(pixels, 50),
            'p75': np.percentile(pixels, 75),
            'p90': np.percentile(pixels, 90),
            'skew': stats.skew(pixels),
            'kurt': stats.kurtosis(pixels)
        }

# analysis per ROI
def calc_intensity_staining(label_id):
    bbox_i = region_labels[label_id][0]

    # update bbox after dilation, used it for cropping the image later
    n_px_dilation = -min(list_erosion)
    y_min, x_min, y_max, x_max = bbox_i
    y_min = max(0, y_min-n_px_dilation)
    y_max = min(y_full, y_max+n_px_dilation)
    x_min = max(0, x_min-n_px_dilation)
    x_max = min(x_full, x_max+n_px_dilation)

    # crop around mask (prevent from calculation in large matrix --> speed up calculation)
    bbox_i = [y_min, x_min, y_max, x_max]
    image_i = crop_image_with_bbox(image, bbox_i)
    mask_i = (masks == label_id).astype(np.uint8)
    mask_i = mask_i[y_min:y_max, x_min:x_max]
    
    # quantification
    cyto2 = erode_mask(mask_i, list_erosion[2])
    cyto1 = erode_mask(mask_i, list_erosion[1])-cyto2
    mem = dilate_mask(mask_i, n_px_dilation)-(cyto1+cyto2)

    